Função do arquivo: Esse arquivo será responsável por consultar os dados, 
construir o dataframe com os dados e enviar os e-mails

Bibliotecas necessarias

In [1]:
# Import da biblioteca pandas que tem como objetivo manipular e acessar
# datasets.
import pandas as pd

# import do submodulo client da biblioteca win32 que tem como objetivo
# possibilitar o controle do ecossistema windows (como outlook, Excel,
# Word e PowerPoint) diretamente através do Python.
import win32com.client as client

# Biblioteca que possibilita a manipulação de datas
import datetime as dt

# Biblioteca que tem como objetivo possibilitar que o python acesse
# e manipule arquivos e diretórios do sistema operacional (sistema 
# de arquivos do SO).
import os

# Função da biblioteca dotenv que tem como objetivo carregar no código
# variáveis de ambientes presentes no arquivo .env
from dotenv import load_dotenv

Acessando as variáveis de ambientes

In [ ]:
# Ira carregar a variável de amboente no código
load_dotenv()

# getenv: Função da biblioteca os que tem como objetivo acessar variáveis
# de ambiente. A função irá receber como argumento o nome da variável
# que deverá ser acessada.
EMAIL = os.getenv('EMAIL')

lendo os dados do banco de dados (JSON)

In [ ]:
# Função da biblioteca pandas que tem como objetivo ler os dados de um
# arquivo/dataset JSON. A função recebe como argumento o caminho do
# banco de dados com a forma que ele deve retornar os dados (arquivo JSON)
# e o orient que serve para indicar como o arquivo json deve retornar 
# os valores.
df_dados = pd.read_json("https://alarme-b3d19-default-rtdb.firebaseio.com/.json", orient='index')

# Após criar o dataframe, vamos acessar a coluna de data e converte-los
# em em formato de datas usando a função to_datetime do pandas.
df_dados['Data'] = pd.to_datetime(df_dados['Data'])

# Ira exibir os dados do dataframe.
display(df_dados)

,Data,Vazao
-OtoU7mjE2h37dyAN9Lq,2026-05-29 13:06:42.099363,117
-OtoUAl40L_nPROe0ura,2026-05-29 13:06:53.314368,124
-OtoUDZTNuVvJvqSCKV3,2026-05-29 13:07:05.489741,159
-OtoUGSQfQjIx4OwS30u,2026-05-29 13:07:16.972128,199
-OtoUJM3z-Uloy3byHAG,2026-05-29 13:07:28.809154,168
...,...,...
-OtpQ5FVgCZqCHkfcgkC,2026-05-29 17:28:40.271307,112
-OtpQ7sBX8lHSzedTQO_,2026-05-29 17:28:51.511296,182
-OtpQqktlzhl58xEiszu,2026-05-29 17:31:55.362165,126
-OtpQtMhT2u4Jcfs9s5H,2026-05-29 17:32:10.190601,125


Estipulando o periodo de análise

In [ ]:
# Este trecho irá definir o tempo que o sistema irá analisar os dados
# coletados. Dessa forma, ele sempre analisará os dados mais recentes
# que chegaram no banco.

# Ira conter a data e a hora atual do sistema
agora = dt.datetime.now()

# A classe timedelta (que significa "variação de tempo" ou "diferença de 
# tempo") serve para definir uma duração. Ela não representa um horário fixo
# no relógio, mas sim uma quantidade de tempo isolada que você quer usar
# para fazer as contas matemáticas (somar ou subtrair). Ela recebe em seu
# construtor o valor da duração.
delta = dt.timedelta(minutes=2)

# Como o python permite operações matematicas com objetos de data e hora,
# vamos subtrair do horário 2 minutos.
tempo_limite = agora - delta

In [ ]:
# Ira conter os dados mais recentes do banco de dados. Para realizar esse filtro
# vamos usar o atributo loc do objeto que recebe como valor, a condição que será aplicada no filtro.
dados_analise = df_dados.loc[df_dados['Data'] > tempo_limite] 

# Irá exibir os dados filtrados.
display(dados_analise)

,Data,Vazao
-OtpQqktlzhl58xEiszu,2026-05-29 17:31:55.362165,126
-OtpQtMhT2u4Jcfs9s5H,2026-05-29 17:32:10.190601,125
-OtpQvyehnjtc-6NBV-R,2026-05-29 17:32:20.860718,178


Filtrando dados na base

In [ ]:
# Agora vamos filtrar todos os valores de dados_analise acima de 150
# (também usando o atributo loc)
aviso = dados_analise.loc[dados_analise['Vazao'] > 150]

# Ira mostrar os valores filtrados.
display(aviso)

,Data,Vazao
-OtpQvyehnjtc-6NBV-R,2026-05-29 17:32:20.860718,178


In [ ]:
# [[]]: Serve para indicar que queremos acessar os valores de duas colunas.

# values: Atributo do objeto retornado pelo pandas (dataframe) que recebe
# os valores das colunas do dataset.

# tolist: Método da biblioteca pandas que transforma os valores em listas.
lista_avisos = aviso[['Data', 'Vazao']].values.tolist()

print(lista_avisos)

[[Timestamp('2026-05-29 17:32:20.860718'), 178]]


Criando o e-mail

In [ ]:
# Dispatch: Metodo do modulo client que indica a ferramenta que queremos
# controlar. Ele retorna um objeto que contém os metodos e atributos 
# necessários para a construção do e-mail.
outlook = client.Dispatch('Outlook.Application')

In [ ]:
# Irá verificar a quantidade de itens da lista de avisos com o
# o objetivo de validar se o sistema encontrou ou não itens acima
# de 150.
if len(lista_avisos) == 0:
    
    # Se essa condição for verdadeira, vamos imprimir essa mensagem e suspender o envio do e-mail.  
    print("A lista está vazia, não há valores para enviar")

else:
    # Caso o sistema encontre valores ácima de 150, vamos iniciar o processo
    # de envio dos dados.
    
    # Como queremos enviar um email para cada valor acima de 150, vamos criar
    # um loop for que irá percorrer a lista de avisos e mandar um email 
    # para cada valor presente na lista.
    for avisos in lista_avisos:
        
        # Ira receber os valores da data
        data_aviso = avisos[0]

        # Ira formatar os valores da data usando a função strftime do pandas
        data_aviso = data_aviso.strftime("%d/%m/%Y %H:%M:%S")

        # Ira conter os valores aleatórios acima de 150 
        vazao_aviso = avisos[1]

        # metodo da classe retornada que tem como objetivo criar 
        # um item em branco para cada mensagem.
        mensagem = outlook.CreateItem(0)

        # Método da classe retornada pelo dispatch que tem como objetivo
        # apresentar ao usuário a caixa de mensagem do outlook com as
        # informações especificadas.
        mensagem.Display()

        # Atributo da classe retornada que possui o endereço de email do
        # destinatário
        mensagem.To = EMAIL

        # Atributo da classe retornada que possui o assunto do e-mail
        mensagem.Subject = "Alerta, valores abaixo de 150"

        # Atributo da classe retornada que possui a mensagem que será
        # enviada
        mensagem.Body = f'''Prezado, no dia / hora {data_aviso} o valor da vazão foi de {vazao_aviso}, superando o valor aceitável'''

        # Método da classe retornada que irá salvar a mensagem
        mensagem.save()

        # Método da classe retornada que enviará a mensagem.
        mensagem.Send()